# 🔍 Transfermarkt Scraper
## Extrae datos de jugadores sudamericanos

Este notebook busca jugadores en Transfermarkt y extrae:
- 💰 Valor de mercado
- 🔗 Link al perfil
- 👔 Agente/representante
- 📅 Fin de contrato
- 🖼️ URL de imagen

---
**Instrucciones:**
1. Ejecutá las celdas en orden (Shift+Enter)
2. En la celda de configuración, ajustá el rango de jugadores
3. Al final se descarga un CSV con los resultados

In [ ]:
# ============================================
# CELDA 1: Instalar dependencias
# ============================================
!pip install requests beautifulsoup4 pandas lxml -q
print("✅ Dependencias instaladas!")

In [ ]:
# ============================================
# CELDA 2: Imports y configuración
# ============================================
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.parse import quote
import random

# Headers para simular navegador real
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'es-ES,es;q=0.9,en;q=0.8',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
}

# Mapeo de ligas sudamericanas
LIGA_MAP = {
    'colombia': 'CO1',
    'liga betplay': 'CO1',
    'categoría primera a': 'CO1',
    '2° de colombia': 'CO2',
    '2° colombia': 'CO2',
    'segunda colombia': 'CO2',
    'torneo betplay': 'CO2',
    'primera b colombia': 'CO2',
    'argentina': 'AR1N',
    'liga profesional': 'AR1N',
    'primera división argentina': 'AR1N',
    '2° de argentina': 'AR2',
    '2° argentina': 'AR2',
    'segunda argentina': 'AR2',
    'primera nacional': 'AR2',
    'uruguay': 'UY1',
    'primera división uruguay': 'UY1',
    'paraguay': 'PAR1',
    'primera división paraguay': 'PAR1',
    'chile': 'CL1',
    'primera división chile': 'CL1',
}

def normalizar_liga(liga_texto):
    """Normaliza el nombre de la liga para buscar en el mapeo"""
    if pd.isna(liga_texto):
        return None
    liga_lower = str(liga_texto).lower().strip()
    for key, value in LIGA_MAP.items():
        if key in liga_lower or liga_lower in key:
            return value
    return None

print("✅ Configuración cargada!")

In [ ]:
# ============================================
# CELDA 3: Funciones de scraping
# ============================================

def buscar_jugador(nombre, equipo=None):
    """Busca un jugador en Transfermarkt y retorna la URL de su perfil"""
    try:
        nombre_limpio = nombre.strip()
        nombre_encoded = quote(nombre_limpio)
        
        search_url = f"https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query={nombre_encoded}"
        
        response = requests.get(search_url, headers=HEADERS, timeout=15)
        
        if response.status_code != 200:
            return None
        
        soup = BeautifulSoup(response.content, 'lxml')
        
        # Buscar tabla de jugadores
        tables = soup.find_all('table', class_='items')
        
        for table in tables:
            rows = table.find_all('tr', class_=['odd', 'even'])
            
            for row in rows:
                link_tag = row.find('a', class_='spielprofil_tooltip')
                if link_tag and link_tag.get('href'):
                    # Si tenemos equipo, intentar matchear
                    if equipo:
                        row_text = row.get_text().lower()
                        equipo_lower = equipo.lower()
                        # Match parcial del equipo
                        equipo_words = equipo_lower.split()
                        if any(word in row_text for word in equipo_words if len(word) > 3):
                            return "https://www.transfermarkt.com" + link_tag['href']
                    
            # Si no matcheó con equipo, devolver el primero
            first_row = rows[0] if rows else None
            if first_row:
                link_tag = first_row.find('a', class_='spielprofil_tooltip')
                if link_tag and link_tag.get('href'):
                    return "https://www.transfermarkt.com" + link_tag['href']
        
        return None
    
    except Exception as e:
        return None


def obtener_datos_jugador(url):
    """Extrae datos del perfil de un jugador"""
    datos = {
        'transfermarkt_url': url,
        'valor_mercado': None,
        'agente': None,
        'fin_contrato': None,
        'imagen_url': None
    }
    
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        
        if response.status_code != 200:
            return datos
        
        soup = BeautifulSoup(response.content, 'lxml')
        
        # 1. Valor de mercado
        market_value = soup.find('a', class_='data-header__market-value-wrapper')
        if market_value:
            datos['valor_mercado'] = market_value.get_text(strip=True)
        
        # 2. Imagen del jugador
        img_tag = soup.find('img', class_='data-header__profile-image')
        if img_tag and img_tag.get('src'):
            datos['imagen_url'] = img_tag['src']
        
        # 3. Buscar en la sección de información
        info_sections = soup.find_all('span', class_='info-table__content')
        labels = soup.find_all('span', class_='info-table__content--regular')
        
        # Método alternativo: buscar todas las filas de info
        info_rows = soup.find_all('li', class_='data-header__label')
        for row in info_rows:
            text = row.get_text(strip=True).lower()
            
            if 'agent' in text:
                link = row.find('a')
                if link:
                    datos['agente'] = link.get_text(strip=True)
            
            if 'contract' in text or 'contrato' in text:
                # Extraer fecha
                match = re.search(r'(\w{3}\s+\d{1,2},\s+\d{4}|\d{1,2}/\d{1,2}/\d{4}|\d{4})', text)
                if match:
                    datos['fin_contrato'] = match.group(1)
        
        # Buscar agente en otra ubicación
        if not datos['agente']:
            agent_span = soup.find('span', string=re.compile(r'Agent', re.I))
            if agent_span:
                parent = agent_span.find_parent('li') or agent_span.find_parent('div')
                if parent:
                    link = parent.find('a')
                    if link:
                        datos['agente'] = link.get_text(strip=True)
        
        # Buscar contrato en header
        if not datos['fin_contrato']:
            contract_elem = soup.find('span', string=re.compile(r'Contract expires', re.I))
            if contract_elem:
                parent = contract_elem.find_parent()
                if parent:
                    full_text = parent.get_text(strip=True)
                    match = re.search(r'(\w{3}\s+\d{1,2},\s+\d{4}|\d{4})', full_text)
                    if match:
                        datos['fin_contrato'] = match.group(1)
        
        return datos
    
    except Exception as e:
        return datos

print("✅ Funciones de scraping listas!")

In [ ]:
# ============================================
# CELDA 4: Cargar CSV de jugadores
# ============================================

CSV_URL = "https://docs.google.com/spreadsheets/d/e/2PACX-1vSneBjGlw2I3SyXV-uw1V8Cs_O4lbiQw39melKEZJNhunpshakPrn7AZQBN2L8N9Yw_HA-EeVOt3qvf/pub?gid=2002620668&single=true&output=csv"

print("📥 Cargando CSV de jugadores...")
df_jugadores = pd.read_csv(CSV_URL)

print(f"\n✅ CSV cargado!")
print(f"📊 Total de jugadores: {len(df_jugadores)}")
print(f"\n📋 Columnas disponibles:")
print(list(df_jugadores.columns[:10]))

print(f"\n👀 Primeros 5 jugadores:")
df_jugadores[['Jugador', 'Liga', 'Equipo']].head()

In [ ]:
# ============================================
# CELDA 5: ⚙️ CONFIGURACIÓN - MODIFICAR AQUÍ
# ============================================

# 🎯 Rango de jugadores a procesar
DESDE_FILA = 0          # Empezar desde esta fila (0 = primero)
HASTA_FILA = 50         # Hasta esta fila (cambiar a len(df_jugadores) para TODOS)

# ⏱️ Pausas entre requests (evitar bloqueos)
PAUSA_MIN = 3           # Segundos mínimos
PAUSA_MAX = 6           # Segundos máximos

# 📝 Nombres de columnas en tu CSV (ajustar si es necesario)
COL_JUGADOR = 'Jugador'
COL_LIGA = 'Liga'
COL_EQUIPO = 'Equipo'

# ============================================
print("⚙️ CONFIGURACIÓN ACTUAL:")
print(f"   Procesar filas: {DESDE_FILA} a {HASTA_FILA}")
print(f"   Total a procesar: {min(HASTA_FILA, len(df_jugadores)) - DESDE_FILA} jugadores")
print(f"   Pausa entre requests: {PAUSA_MIN}-{PAUSA_MAX} segundos")
tiempo_estimado = ((min(HASTA_FILA, len(df_jugadores)) - DESDE_FILA) * (PAUSA_MIN + PAUSA_MAX) / 2) / 60
print(f"   ⏱️ Tiempo estimado: ~{tiempo_estimado:.0f} minutos")

In [ ]:
# ============================================
# CELDA 6: 🚀 EJECUTAR SCRAPING
# ============================================

resultados = []
errores = []

total = min(HASTA_FILA, len(df_jugadores)) - DESDE_FILA
print(f"🚀 Iniciando scraping de {total} jugadores...")
print("=" * 60)

for idx in range(DESDE_FILA, min(HASTA_FILA, len(df_jugadores))):
    row = df_jugadores.iloc[idx]
    nombre = row[COL_JUGADOR]
    liga = row.get(COL_LIGA, '')
    equipo = row.get(COL_EQUIPO, '')
    
    progreso = idx - DESDE_FILA + 1
    print(f"[{progreso}/{total}] {nombre} ({equipo})...", end=" ")
    
    try:
        # Buscar jugador
        url = buscar_jugador(nombre, equipo)
        
        if url:
            print("✅ Encontrado!", end=" ")
            datos = obtener_datos_jugador(url)
            datos['jugador_original'] = nombre
            datos['equipo_original'] = equipo
            datos['liga_original'] = liga
            resultados.append(datos)
            
            valor = datos['valor_mercado'] or 'N/A'
            print(f"→ {valor}")
        else:
            print("❌ No encontrado")
            resultados.append({
                'jugador_original': nombre,
                'equipo_original': equipo,
                'liga_original': liga,
                'transfermarkt_url': None,
                'valor_mercado': None,
                'agente': None,
                'fin_contrato': None,
                'imagen_url': None
            })
    
    except Exception as e:
        print(f"⚠️ Error: {e}")
        errores.append({'jugador': nombre, 'error': str(e)})
        resultados.append({
            'jugador_original': nombre,
            'equipo_original': equipo,
            'liga_original': liga,
            'transfermarkt_url': None,
            'valor_mercado': None,
            'agente': None,
            'fin_contrato': None,
            'imagen_url': None
        })
    
    # Pausa aleatoria
    time.sleep(random.uniform(PAUSA_MIN, PAUSA_MAX))

print("=" * 60)
print(f"\n✅ COMPLETADO!")
print(f"   Procesados: {len(resultados)}")
print(f"   Errores: {len(errores)}")

In [ ]:
# ============================================
# CELDA 7: 📊 Ver resultados
# ============================================

df_resultados = pd.DataFrame(resultados)

# Reordenar columnas
columnas_orden = ['jugador_original', 'equipo_original', 'liga_original',
                  'valor_mercado', 'transfermarkt_url', 'agente',
                  'fin_contrato', 'imagen_url']
df_resultados = df_resultados[columnas_orden]

# Estadísticas
encontrados = df_resultados['transfermarkt_url'].notna().sum()
no_encontrados = df_resultados['transfermarkt_url'].isna().sum()

print("📊 ESTADÍSTICAS:")
print(f"   ✅ Encontrados: {encontrados} ({encontrados/len(df_resultados)*100:.1f}%)")
print(f"   ❌ No encontrados: {no_encontrados} ({no_encontrados/len(df_resultados)*100:.1f}%)")

print("\n📋 Muestra de resultados:")
df_resultados.head(10)

In [ ]:
# ============================================
# CELDA 8: 💾 Guardar y descargar CSV
# ============================================

nombre_archivo = f'transfermarkt_jugadores_{DESDE_FILA}_{HASTA_FILA}.csv'

# Guardar CSV
df_resultados.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
print(f"💾 Archivo guardado: {nombre_archivo}")

# Descargar
from google.colab import files
files.download(nombre_archivo)
print("\n📥 Descargando archivo...")

In [ ]:
# ============================================
# CELDA 9 (OPCIONAL): Ver jugadores no encontrados
# ============================================

print("❌ Jugadores NO encontrados (para buscar manualmente):")
print("=" * 50)

no_encontrados_df = df_resultados[df_resultados['transfermarkt_url'].isna()]
for idx, row in no_encontrados_df.iterrows():
    print(f"• {row['jugador_original']} - {row['equipo_original']}")